<a href="https://colab.research.google.com/github/Pandabear-67/Collision-tracker/blob/main/preprocessing/Image_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
import shutil
import json
from sklearn.model_selection import train_test_split

base_path = "/content/drive/MyDrive/accident_files/sim_dataset"

In [ ]:
#check the column types
selected_videos = pd.read_csv('/content/drive/MyDrive/accident_files/selected_videos.csv')
selected_videos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   rgb_path                  245 non-null    object 
 1   annotations_path          245 non-null    object 
 2   type                      245 non-null    object 
 3   accident_time             245 non-null    float64
 4   accident_frame            245 non-null    int64  
 5   center_x                  245 non-null    float64
 6   center_y                  245 non-null    float64
 7   x1                        245 non-null    float64
 8   y1                        245 non-null    float64
 9   x2                        245 non-null    float64
 10  y2                        245 non-null    float64
 11  map                       245 non-null    object 
 12  weather                   245 non-null    object 
 13  camera_position           245 non-null    int64  
 14  no_frames 

In [ ]:
selected_videos = pd.read_csv('/content/drive/MyDrive/accident_files/selected_videos.csv')

# First split: 70% train, 30% temp (for validation and test)
train_df, temp_df = train_test_split(selected_videos, test_size=0.3, shuffle=True, random_state=42, stratify=selected_videos['weather'])

# Second split: 50% from temp_df for validation, 50% for test (results in 15% val, 15% test from original)
val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=True, random_state=42, stratify=temp_df['weather'])

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

selected_videos = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Create directories for the splits if they don't exist
for split_name in ['train', 'val', 'test']:
    os.makedirs(os.path.join(base_path, 'images_without_accident', split_name), exist_ok=True)

for _, video in selected_videos.iterrows():
    offset = video['annotations_start_offset']
    video_name = video['rgb_path'].split('/')[-1].split('.')[0]
    json_path = os.path.join(base_path, "video_annotations", f"{video_name}.json",  f"{video_name}.json")
    annotation = None

    with open(json_path, "r") as f:
      annotation = json.load(f)

    # Get the accident_frame for the current video
    accident_frame = video['accident_frame']

    for frame in annotation["base"]:
      frame_num = frame['iteration'] - offset
      # Stop processing frames for this video if the accident frame is reached or exceeded
      if frame_num >= accident_frame:
        break

      image_path = f"/content/drive/MyDrive/accident_files/random_extracted_frames/frames/{video_name}_frame_{frame_num:0>4}.jpg"
      new_image_path = f"{base_path}/images_without_accident/{video['split']}/{video_name}_frame_{frame_num:0>4}.jpg"
      try:
        shutil.copy(image_path,new_image_path)
      except FileNotFoundError:
        print(f"Skipping file not found: {image_path}")
        continue
print('completed')

Skipping file not found: /content/drive/MyDrive/accident_files/random_extracted_frames/frames/Town07_single_sunset_29_frame_0000.jpg
completed
